Note that only 20 images were used (16 train, 4 test) for demonstration purposes only. The actual dataset is too large to be shown concisely (6000+ frames). 

In [118]:
#Smaller Inputs Used for Demo Purposes

from pyexpat import model
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import glob
import os
import locale
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Add, ZeroPadding2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import concatenate
from tensorflow.keras.metrics import Precision, Recall, BinaryAccuracy
from sklearn import preprocessing
import locale
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from deepforest import CascadeForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras.layers import DepthwiseConv2D
from tensorflow.keras.layers import ReLU, AvgPool2D
import random

gasdata = pd.read_csv('/Users/eddie/Downloads/TrainGasSensorMeasurements.csv')
data_dir = '/Users/eddie/Downloads/NoGas/'
#load the image files
imagedata = sorted(os.listdir(data_dir))
print(len(imagedata))
X_data = []
DEMO_CAP = 10
index = 1
for image in imagedata:
        if index<=10 or index>=6391:
                img = mpimg.imread('/Users/eddie/Downloads/NoGas/'+image)
                img.resize(224,224,3)
                img = img/255.0
                X_data.append(img)
        index = index+1
images = np.array(X_data)
#store loaded images
print(len(images))
gasdata = pd.concat([gasdata.iloc[:10], gasdata.iloc[6390:]])
print(gasdata)
trainAttrX, testAttrX, trainImagesX, testImagesX = train_test_split(gasdata, images, test_size=0.2)
#load the training labels
trainy = trainAttrX["Gas"]
testy = testAttrX["Gas"]
#left with only gas data
trainAttrX = trainAttrX.drop(columns=['Gas'])
testAttrX = testAttrX.drop(columns=['Gas'])
#normalize the gas data
trainAttrX= (trainAttrX - np.min(trainAttrX)) / (np.max(trainAttrX) - np.min(trainAttrX))
testAttrX= (testAttrX - np.min(testAttrX)) / (np.max(testAttrX) - np.min(testAttrX))
print(trainAttrX)


6400
20
      MQ2  MQ7  Gas
0     555  666    0
1     555  666    0
2     556  666    0
3     556  665    0
4     556  665    0
5     557  665    0
6     556  665    0
7     556  665    0
8     556  665    0
9     556  665    0
6390  688  428    1
6391  686  426    1
6392  683  424    1
6393  679  419    1
6394  658  417    1
6395  658  414    1
6396  650  411    1
6397  630  407    1
6398  632  405    1
6399  633  402    1
           MQ2       MQ7
9     0.007519  0.996169
8     0.007519  0.996169
6394  0.774436  0.045977
6398  0.578947  0.000000
6391  0.984962  0.080460
5     0.015038  0.996169
6     0.007519  0.996169
6395  0.774436  0.034483
4     0.007519  0.996169
0     0.000000  1.000000
6390  1.000000  0.088123
6393  0.932331  0.053640
6392  0.962406  0.072797
6396  0.714286  0.022989
7     0.007519  0.996169
3     0.007519  0.996169


In [119]:
#CNN involving only the first three layers of the AlexNet
def alexnet_l3():
    model = Sequential()
    model.add(Conv2D(filters = 96, input_shape = (224, 224, 3),  
                kernel_size = (11, 11), strides = (4, 4),  
                padding = 'valid',kernel_regularizer = 'l2')) 
    model.add(Activation('relu')) 
    # Max-Pooling  
    model.add(MaxPooling2D(pool_size = (2, 2), 
                strides = (2, 2), padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Convolutional Layer 
    model.add(Conv2D(filters = 256, kernel_size = (11, 11),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Max-Pooling 
    model.add(MaxPooling2D(pool_size = (2, 2), strides = (2, 2),  
                padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 3rd Convolutional Layer 
    model.add(Conv2D(filters = 384, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    # Flattening 
    model.add(Flatten()) 
    
    # 1st Dense Layer 
    model.add(Dropout(0.4)) 
    model.add(Dense(4096,kernel_regularizer='l2')) 
    model.add(Activation('relu') )
    # Add Dropout to prevent overfitting 
    model.add(Dropout(0.4)) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Dense Layer 
    model.add(Dense(2048,kernel_regularizer='l2') )
    model.add(Activation('relu')) 
    # Add Dropout 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    model.add(Dense(1,activation = 'sigmoid'))
    return model

#CNN involving only the first four layers of the AlexNet
def alexnet_l4():
    model = Sequential()
    model.add(Conv2D(filters = 96, input_shape = (224, 224, 3),  
                kernel_size = (11, 11), strides = (4, 4),  
                padding = 'valid',kernel_regularizer = 'l2')) 
    model.add(Activation('relu')) 
    # Max-Pooling  
    model.add(MaxPooling2D(pool_size = (2, 2), 
                strides = (2, 2), padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Convolutional Layer 
    model.add(Conv2D(filters = 256, kernel_size = (11, 11),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Max-Pooling 
    model.add(MaxPooling2D(pool_size = (2, 2), strides = (2, 2),  
                padding = 'valid')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 3rd Convolutional Layer 
    model.add(Conv2D(filters = 384, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 4th Convolutional Layer 
    model.add(Conv2D(filters = 384, kernel_size = (3, 3),  
                strides = (1, 1), padding = 'valid')) 
    model.add(Activation('relu')) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # Flattening 
    model.add(Flatten()) 
    
    # 1st Dense Layer 
    model.add(Dropout(0.4)) 
    model.add(Dense(4096,kernel_regularizer='l2')) 
    model.add(Activation('relu') )
    # Add Dropout to prevent overfitting 
    model.add(Dropout(0.4)) 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    
    # 2nd Dense Layer 
    model.add(Dense(2048,kernel_regularizer='l2') )
    model.add(Activation('relu')) 
    # Add Dropout 
    # Batch Normalisation 
    model.add(BatchNormalization()) 
    model.add(Dense(1,activation = 'sigmoid'))
    return model

In [120]:
#CNN involving only the first three blocks of the ResNet
def identity_block(X, f, filters, stage, block):
   
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'
    F1, F2, F3 = filters

    X_shortcut = X
   
    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2a')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding='same', name=conv_name_base + '2b')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2c')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)
    X = Add()([X, X_shortcut])# SKIP Connection
    X = Activation('relu')(X)

    return X

def convolutional_block(X, f, filters, stage, block, s=2):
   
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv2D(filters=F1, kernel_size=(1, 1), strides=(s, s), padding='valid', name=conv_name_base + '2a')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv2D(filters=F2, kernel_size=(f, f), strides=(1, 1), padding='same', name=conv_name_base + '2b')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)
    X = Conv2D(filters=F3, kernel_size=(1, 1), strides=(1, 1), padding='valid', name=conv_name_base + '2c')(X)
    X = BatchNormalization(axis=3, name=bn_name_base + '2c')(X)
    X_shortcut = Conv2D(filters=F3, kernel_size=(1, 1), strides=(s, s), padding='valid', name=conv_name_base + '1',)(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=bn_name_base + '1')         (X_shortcut)

    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X

def ResNet50(input_shape=(224, 224, 3)):

    X_input = Input(input_shape)

    X = ZeroPadding2D((3, 3))(X_input)

    X = Conv2D(64, (7, 7), strides=(2, 2), name='conv1', )(X)
    X = BatchNormalization(axis=3, name='bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, block='a', s=1)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')


    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')
    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

    X = AveragePooling2D(pool_size=(2, 2), padding='same')(X)
    headModel = Flatten()(X)
    headModel=Dense(256, activation='relu', name='fc1')(headModel)
    headModel=Dense(128, activation='relu', name='fc2')(headModel)
    headModel = Dense( 1, name='fc3')(headModel)
    model = Model(inputs=X_input, outputs=headModel, name='ResNet50')
    return model

In [121]:
#CNN involving the full VggNet
def create_cnn_vggnet():
    model = Sequential()
    model.add(Conv2D(input_shape=(224,224,3),filters=64,kernel_size=(3,3),padding="same", activation="relu"))
    model.add(Conv2D(filters=64,kernel_size=(3,3),padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Flatten())
    model.add(Dense(units=4096,activation="relu"))
    model.add(Dense(units=4096,activation="relu"))
    model.add(Dense(units=1, activation="sigmoid"))
    return model

#CNN built using the first 12 layers of the VggNet
def create_cnn_vggnet_12():
    model = Sequential()
    model.add(Conv2D(input_shape=(224,224,3),filters=64,kernel_size=(3,3),padding="same", activation="relu"))
    model.add(Conv2D(filters=64,kernel_size=(3,3),padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    # model.add(Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"))
    model.add(MaxPool2D(pool_size=(2,2),strides=(2,2)))
    model.add(Flatten())
    model.add(Dense(units=4096,activation="relu"))
    model.add(Dense(units=4096,activation="relu"))
    model.add(Dense(units=1, activation="sigmoid"))
    return model

In [122]:
#CNN built using the first seven layers of the EfficientNet
def get_top(x_input):
    """Block top operations
    This functions apply Batch Normalization and Leaky ReLU activation to the input.

    # Arguments:
        x_input: Tensor, input to apply BN and activation  to.

    # Returns:
        Output tensor
    """
    
    x = tf.keras.layers.BatchNormalization()(x_input)
    x = tf.keras.layers.LeakyReLU()(x)
    return x

def get_block(x_input, input_channels, output_channels):
    """MBConv block
    This function defines a mobile Inverted Residual Bottleneck block with BN and Leaky ReLU

    # Arguments
        x_input: Tensor, input tensor of conv layer.
        input_channels: Integer, the dimentionality of the input space.
        output_channels: Integer, the dimensionality of the output space.
            
    # Returns
        Output tensor.
    """

    x = tf.keras.layers.Conv2D(input_channels, kernel_size=(1, 1), padding='same', use_bias=False)(x_input)
    x = get_top(x)
    x = tf.keras.layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False)(x)
    x = get_top(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 1), strides=(2, 1))(x)
    x = tf.keras.layers.DepthwiseConv2D(kernel_size=(3, 1), padding='same', use_bias=False)(x)
    x = get_top(x)
    x = tf.keras.layers.Conv2D(output_channels, kernel_size=(2, 1), strides=(1, 2), padding='same', use_bias=False)(x)
    return x


def create_cnn_efficientnet_7(plot_model=False):
    """EffNet
    This function defines a EfficientNet architecture.

    # Arguments
        input_shape: An integer or tuple/list of 3 integers, shape
            of input tensor.
        num_classes: Integer, number of classes.
        plot_model: Boolean, whether to plot model architecture or not
    # Returns
        EfficientNet model.
    """
    x_input = tf.keras.layers.Input(shape=(224,224,3))
    x = get_block(x_input, 32, 64)
    x = get_block(x, 64, 128)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.models.Model(inputs=x_input, outputs=x)

    return model
#create CNN using first 3 layers of the EfficientNet 
def get_top(x_input):
    """Block top operations
    This functions apply Batch Normalization and Leaky ReLU activation to the input.

    # Arguments:
        x_input: Tensor, input to apply BN and activation  to.

    # Returns:
        Output tensor
    """
    
    x = tf.keras.layers.BatchNormalization()(x_input)
    x = tf.keras.layers.LeakyReLU()(x)
    return x

def get_block(x_input, input_channels, output_channels):
    """MBConv block
    This function defines a mobile Inverted Residual Bottleneck block with BN and Leaky ReLU

    # Arguments
        x_input: Tensor, input tensor of conv layer.
        input_channels: Integer, the dimentionality of the input space.
        output_channels: Integer, the dimensionality of the output space.
            
    # Returns
        Output tensor.
    """

    x = tf.keras.layers.Conv2D(input_channels, kernel_size=(1, 1), padding='same', use_bias=False)(x_input)
    x = get_top(x)
    x = tf.keras.layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False)(x)
    x = get_top(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 1), strides=(2, 1))(x)
    x = tf.keras.layers.DepthwiseConv2D(kernel_size=(3, 1), padding='same', use_bias=False)(x)
    x = get_top(x)
    x = tf.keras.layers.Conv2D(output_channels, kernel_size=(2, 1), strides=(1, 2), padding='same', use_bias=False)(x)
    return x


def create_cnn_efficientnet_3(plot_model=False):
    """EffNet
    This function defines a EfficientNet architecture.

    # Arguments
        input_shape: An integer or tuple/list of 3 integers, shape
            of input tensor.
        num_classes: Integer, number of classes.
        plot_model: Boolean, whether to plot model architecture or not
    # Returns
        EfficientNet model.
    """
    x_input = tf.keras.layers.Input(shape=(224,224,3))
    x = get_block(x_input, 32, 64)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.models.Model(inputs=x_input, outputs=x)

    return model



In [123]:
#create CNN using first 6 blocks of the MobileNet
def mobilnet_block (x, filters, strides):
    
    x = DepthwiseConv2D(kernel_size = 3, strides = strides, padding = 'same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    
    x = Conv2D(filters = filters, kernel_size = 1, strides = 1)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    
    return x
def create_cnn_mobilenet():
    input = Input(shape = (224,224,3))
    x = Conv2D(filters = 32, kernel_size = 3, strides = 2, padding = 'same')(input)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # main part of the model
    x = mobilnet_block(x, filters = 64, strides = 1)
    x = mobilnet_block(x, filters = 128, strides = 2)
    x = mobilnet_block(x, filters = 128, strides = 1)
    x = mobilnet_block(x, filters = 256, strides = 2)
    x = mobilnet_block(x, filters = 256, strides = 1)
    x = mobilnet_block(x, filters = 512, strides = 2)
    x = AvgPool2D (pool_size = 7, strides = 1, data_format='channels_first')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    output= Flatten()(x)
    output = Dense (4096, activation = "relu",kernel_regularizer='l2')(output)
    output = (Dropout(0.4))(output)
    output = Dense (2048, activation = "relu",kernel_regularizer='l2')(output)
    output = Dense (1, activation = "sigmoid")(output)
    model = Model(inputs=input, outputs=output)
    return model

In [124]:
#predict the train images with the three most optimal CNNs as described in paper
#the classfifiers can be replaced if necessary with other CNN options
from tensorflow.keras.models import load_model
clf1 = load_model("/Users/eddie/Downloads/ModifiedDFC/CNNs/efficientnet.keras")
clf1predict = clf1.predict(trainImagesX)

clf2 = create_cnn_vggnet_12()
clf2 = load_model("/Users/eddie/Downloads/ModifiedDFC/CNNs/vggnet.keras")
clf2predict = clf2.predict(trainImagesX)

clf3 = ResNet50()
clf3 = load_model("/Users/eddie/Downloads/ModifiedDFC/CNNs/ResNet50.keras")
clf3predict = clf3.predict(trainImagesX)


clf1predict = np.array(clf1predict)
clf2predict = np.array(clf2predict)
clf3predict = np.array(clf3predict)

#concatenate together training data
trainAttrX = np.array(trainAttrX)
trainAttrX.resize(16,2)
trainData = concatenate([clf1predict,clf2predict, clf3predict, trainAttrX])
trainData = np.array(trainData)
trainData.reshape(-1,1)

print(trainData.shape)


(16, 5)


In [125]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from mondrianforest import MondrianForestClassifier


In [126]:
#load the optimized DFC model
model = CascadeForestClassifier(n_tolerant_rounds=3, max_layers=20, backend='sklearn')

n_estimators = 2  # the number of base estimators per cascade layer
estimators = [ExtraTreesClassifier(n_estimators = 107, min_samples_split=7,min_samples_leaf=13,random_state=1),AdaBoostClassifier(n_estimators=90,learning_rate=0.417,random_state=2),XGBClassifier(eta = 1.0, max_depth=11),MondrianForestClassifier(n_tree = 2)]
model.set_estimator(estimators)

In [127]:
print(trainData)
print(trainy)

[[2.2412837e-03 7.2210133e-02 5.8196473e+00 7.5187972e-03 9.9616855e-01]
 [4.4217706e-04 6.4096600e-02 5.4485850e+00 7.5187972e-03 9.9616855e-01]
 [1.0000000e+00 9.9773103e-01 9.0929546e+00 7.7443612e-01 4.5977011e-02]
 [9.9999988e-01 9.9713206e-01 1.0583515e+01 5.7894737e-01 0.0000000e+00]
 [1.0000000e+00 9.9784273e-01 7.3220220e+00 9.8496240e-01 8.0459774e-02]
 [2.1406257e-01 9.6312302e-01 7.4912338e+00 1.5037594e-02 9.9616855e-01]
 [1.1325181e-03 5.7065457e-02 6.2600846e+00 7.5187972e-03 9.9616855e-01]
 [1.0000000e+00 9.9773002e-01 8.9325542e+00 7.7443612e-01 3.4482758e-02]
 [6.2839150e-02 8.9829731e-01 7.4112229e+00 7.5187972e-03 9.9616855e-01]
 [1.4625788e-03 7.5348943e-02 6.9589243e+00 0.0000000e+00 1.0000000e+00]
 [9.9999154e-01 9.9153197e-01 9.3656683e+00 1.0000000e+00 8.8122606e-02]
 [1.0000000e+00 9.9771309e-01 8.4461336e+00 9.3233085e-01 5.3639848e-02]
 [1.0000000e+00 9.9786031e-01 7.4287853e+00 9.6240604e-01 7.2796933e-02]
 [1.0000000e+00 9.9767804e-01 8.6532822e+00 7.14285

In [128]:
#train the model
model.fit(trainData,trainy)

[2026-01-05 08:41:27.597] Start to fit the model:
[2026-01-05 08:41:27.597] Fitting cascade layer = 0 


c:\Users\eddie\anaconda3\envs\ModifiedDFC\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[00:41:27] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:27] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:27] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:27] WARNING: C:/Users/Administrator/workspace/xgboost-win64_rel

c:\Users\eddie\anaconda3\envs\ModifiedDFC\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[00:41:28] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:28] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:28] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:28] WARNING: C:/Users/Administrator/workspace/xgboost-win64_rel

...

c:\Users\eddie\anaconda3\envs\ModifiedDFC\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_rel

c:\Users\eddie\anaconda3\envs\ModifiedDFC\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_release_1.5.1/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.
[00:41:29] WARNING: C:/Users/Administrator/workspace/xgboost-win64_rel

c:\Users\eddie\anaconda3\envs\ModifiedDFC\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


In [129]:
clf1predictTest = np.array(clf1.predict(testImagesX))
clf2predictTest = np.array(clf2.predict(testImagesX))
clf3predictTest = np.array(clf3.predict(testImagesX))

print(clf1predictTest.shape)
print(clf2predictTest.shape)

print(testAttrX)

#prepare the test data
testData = concatenate([clf1predictTest, clf2predictTest, clf3predictTest, testAttrX])
print(testData)
print(testData.shape)


(4, 1)
(4, 1)
           MQ2       MQ7
2     0.012821  1.000000
1     0.000000  1.000000
6399  1.000000  0.000000
6397  0.961538  0.018939
tf.Tensor(
[[1.22244358e-02 5.22893310e-01 6.64309359e+00 1.28205130e-02
  1.00000000e+00]
 [2.28479505e-03 4.75460291e-02 6.63295650e+00 0.00000000e+00
  1.00000000e+00]
 [9.99999523e-01 9.95753646e-01 1.05924425e+01 1.00000000e+00
  0.00000000e+00]
 [1.00000000e+00 9.97601509e-01 8.36434174e+00 9.61538434e-01
  1.89393945e-02]], shape=(4, 5), dtype=float32)
(4, 5)


In [130]:
#Evaluate Model Performance
performance = model.predict(testData)
print(performance)
actual = []
for value in performance: 
    #print(value)
    if(value>=0.5):
            actual.append(1)
    else:
            actual.append(0)
print(actual)
# print(result)
accuracy = accuracy_score(testy, actual)
print('Accuracy: %f' % accuracy)
precision = precision_score(testy, actual)
print('Precision: %f' % precision)
recall = recall_score(testy, actual)
print('Recall: %f' % recall)
f1 = f1_score(testy, actual)
print('F1 score: %f' % f1)

[2026-01-05 08:41:37.902] Start to evalute the model:
[2026-01-05 08:41:37.902] Evaluating cascade layer = 0 
[2026-01-05 08:41:37.937] Evaluating cascade layer = 1 
[2026-01-05 08:41:37.980] Evaluating cascade layer = 2 
[0 0 1 1]
[0, 0, 1, 1]
Accuracy: 1.000000
Precision: 1.000000
Recall: 1.000000
F1 score: 1.000000
